In [1]:
import os
import sys
import shutil
import duckdb
import requests
import zipfile
import io
import time
import logging
import re
import warnings
import gc
import unicodedata
import pandas as pd
from bs4 import BeautifulSoup as soup
from urllib.parse import urljoin

from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql.functions import col, lit, row_number, regexp_replace, trim, upper, when, to_date, monotonically_increasing_id, to_timestamp
from pyspark.sql.types import IntegerType
from delta.tables import DeltaTable

# Configuração de Logging e Warnings
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
warnings.filterwarnings('ignore', category=pd.errors.ParserWarning)

# ==============================================================================
# 0. CONTROLE DE ESTADO (CHECKPOINT)
# ==============================================================================
class ProcessTracker:
    """Gerencia quais URLs já foram processadas com sucesso para evitar retrabalho."""
    def __init__(self, tracking_file="processed_files.txt"):
        self.tracking_file = tracking_file
        self.processed = self._load_processed()

    def _load_processed(self):
        if not os.path.exists(self.tracking_file):
            return set()
        with open(self.tracking_file, "r") as f:
            return set(line.strip() for line in f)

    def is_processed(self, url):
        return url in self.processed

    def mark_processed(self, url):
        with open(self.tracking_file, "a") as f:
            f.write(f"{url}\n")
        self.processed.add(url)

# ==============================================================================
# 1. CLASSES DE INFRAESTRUTURA SPARK
# ==============================================================================
class SparkFactory:
    @staticmethod
    def get_session(app_name="IntegratedETL"):
        existing = SparkSession.getActiveSession()
        if existing:
            print("⚠️ Parando SparkSession existente para aplicar novas configurações...")
            existing.stop()

        print("🚀 Iniciando nova SparkSession com configurações otimizadas...")
        return (SparkSession.builder.appName(app_name).master("local[*]")
                # MEMÓRIA
                .config("spark.driver.memory", "32g")  # Ajustado para evitar travar a máquina inteira se for desktop
                .config("spark.executor.memory", "32g")
                # PERFORMANCE
                .config("spark.sql.shuffle.partitions", "200")
                .config("spark.driver.maxResultSize", "8g")
                .config("spark.memory.fraction", "0.6") # Reduzido levemente para deixar mais heap para execução
                .config("spark.sql.parquet.compression.codec", "snappy")
                # DELTA
                .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0")
                .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
                .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
                .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
                .getOrCreate())

class Sanitizer:
    @staticmethod
    def clean_generic(df: DataFrame) -> DataFrame:
        for field in df.schema.fields:
            if str(field.dataType) == "StringType":
                df = df.withColumn(field.name, when(col(field.name).isNull(), lit("")).otherwise(upper(trim(col(field.name)))))
        return df
    @staticmethod
    def only_numbers(df: DataFrame, cols: list) -> DataFrame:
        cols_lower = [c.lower() for c in cols]
        for c in df.columns:
            if c.lower() in cols_lower:
                df = df.withColumn(c, regexp_replace(col(c), "[^0-9]", ""))
        return df

class DeltaManager:
    def __init__(self, spark: SparkSession, base_path: str):
        self.spark = spark; self.base_path = base_path

    def log_stats(self, table_name: str):
        try:
            path = f"{self.base_path}/{table_name}"
            dt = DeltaTable.forPath(self.spark, path)
            # Operação leve para contar, evitando count() total demorado em big data se não necessário
            # total_count = dt.toDF().count() 
            # print(f"    📈 Stats: {total_count} registros totais.") 
            # (Comentado o count total pois em 118M de linhas isso demora muito. Focando nas métricas do log)
            
            last_op = dt.history(1).select("operation", "operationMetrics").collect()[0]
            metrics = last_op["operationMetrics"] if last_op["operationMetrics"] else {}
            
            if last_op["operation"] == "MERGE":
                inserted = metrics.get("numTargetRowsInserted", "0")
                updated = metrics.get("numTargetRowsUpdated", "0")
                print(f"    📈 Stats (Merge): +{inserted} ins, ↻{updated} upd")
            else:
                written = metrics.get("numOutputRows", "0")
                print(f"    📈 Stats (Write): {written} linhas processadas nesta rodada.")
        except Exception as e:
            print(f"    ⚠️  Stats indisponíveis: {e}")

    def upsert_dimension(self, df_source: DataFrame, table_name: str, keys: list, id_col="ID"):
        path = f"{self.base_path}/{table_name}"
        
        # Criação
        if not os.path.exists(path) and not os.path.exists(path + "/_delta_log"):
            w = Window.partitionBy(lit(1)).orderBy(*[col(k) for k in keys])
            df_final = df_source.withColumn(id_col, row_number().over(w).cast(IntegerType()))
            (df_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path))
            self.log_stats(table_name)
            return self.spark.read.format("delta").load(path)

        # Schema Evolution
        delta_table = DeltaTable.forPath(self.spark, path)
        target_cols = [c.lower() for c in delta_table.toDF().columns]
        missing = [c for c in df_source.columns if c.lower() not in target_cols]
        if missing:
            (df_source.limit(0).write.format("delta").mode("append").option("mergeSchema", "true").save(path))
            delta_table = DeltaTable.forPath(self.spark, path)

        # Upsert Logic
        try: max_id = delta_table.toDF().agg({f"`{id_col}`": "max"}).collect()[0][0] or 0
        except: max_id = 0
        
        w_new = Window.partitionBy(lit(1)).orderBy(*[col(k) for k in keys])
        df_updates = df_source.withColumn("New_ID_Gen", row_number().over(w_new) + max_id)
        
        match_cond = " AND ".join([f"target.`{k}` = source.`{k}`" for k in keys])
        update_cols = {c: f"source.`{c}`" for c in df_source.columns if c not in keys and c != id_col and c != "New_ID_Gen"}
        insert_cols = {c: f"source.`{c}`" for c in df_source.columns if c != "New_ID_Gen"}
        insert_cols[id_col] = "source.New_ID_Gen"

        (delta_table.alias("target").merge(df_updates.alias("source"), match_cond)
         .whenMatchedUpdate(set=update_cols).whenNotMatchedInsert(values=insert_cols).execute())
        
        self.log_stats(table_name)
        return self.spark.read.format("delta").load(path)

class GenericETLEngine:
    def __init__(self, base_output_dir="data/delta"):
        self.spark = SparkFactory.get_session()
        self.delta_mgr = DeltaManager(self.spark, base_output_dir)

    def _apply_mapping(self, df, mapping):
        select_exprs = []
        for src, tgt in mapping.items():
            if src.startswith("LIT_"):
                val = ""
                if src == "LIT_NIVEL_DETALHE_COMPLETO": val = "DETALHADO"
                elif src == "LIT_NIVEL_DETALHE_SIMPLES": val = "MUNICIPIO_UF"
                elif src == "LIT_NAO_SE_APLICA": val = "N/A"
                # Defaults vazios
                elif src.startswith("LIT_STRING_VAZIA"): val = ""
                
                select_exprs.append(lit(val).alias(tgt))
                continue
            
            if src not in df.columns:
                col_expr = lit(None)
            else:
                col_expr = col(src)

            if isinstance(tgt, tuple):
                if len(tgt) == 3: name, type_, fmt = tgt
                else: name, type_ = tgt; fmt = None
                
                if type_ == "date": select_exprs.append(to_date(col_expr, fmt).alias(name))
                elif type_ == "timestamp": select_exprs.append(to_timestamp(col_expr, fmt).alias(name))
                elif type_ == "int": select_exprs.append(col_expr.cast("int").alias(name))
                elif type_ == "decimal": select_exprs.append(col_expr.cast("decimal(30,10)").alias(name))
                elif type_ == "latlong": select_exprs.append(regexp_replace(col_expr, ",", ".").cast("decimal(12,10)").alias(name))
            else:
                select_exprs.append(col_expr.alias(tgt))
        return df.select(*select_exprs)

    def run(self, config: dict):
        print(f"\n🚀 Pipeline Spark: {config['name']}")
        
        try:
            df_raw = self.spark.read.format("csv").option("header", "true").option("sep", ";").load(config['source']['path'])
        except Exception as e:
            print(f"⚠️ Falha ao ler arquivo temporário no Spark: {e}")
            return

        df_clean = Sanitizer.clean_generic(df_raw)
        if "clean_numbers" in config: df_clean = Sanitizer.only_numbers(df_clean, config["clean_numbers"])

        loaded_dims = {}

        # 1. Dimensões
        for dim_conf in config.get("dimensions", []):
            table_name = dim_conf["target_table"]
            print(f"🧩 Dimensão: {table_name}")
            df_dim_source = self._apply_mapping(df_clean, dim_conf["mapping"])
            
            if "lookups" in dim_conf:
                for dim_ref, info in dim_conf["lookups"].items():
                    if dim_ref in loaded_dims:
                        df_ref = loaded_dims[dim_ref]["df"]
                        join_keys = info["join_keys"]; ref_pk = info.get("ref_pk", "ID"); target_fk = info["target_fk"]
                        valid_keys = [k for k in join_keys if k in df_dim_source.columns]
                        if not valid_keys: continue

                        df_ref_sub = df_ref.select(*valid_keys, ref_pk)
                        df_dim_source = df_dim_source.join(df_ref_sub, on=valid_keys, how="left")
                        if ref_pk in df_dim_source.columns: df_dim_source = df_dim_source.withColumnRenamed(ref_pk, target_fk)
                        df_dim_source = df_dim_source.drop(*valid_keys)

            keys = dim_conf["keys"]; id_col = dim_conf.get("id_col", "ID")
            for k in keys:
                if k in df_dim_source.columns: df_dim_source = df_dim_source.filter(col(k).isNotNull())
            
            if all(k in df_dim_source.columns for k in keys):
                df_dim_source = df_dim_source.dropDuplicates(subset=keys)
                df_dim_updated = self.delta_mgr.upsert_dimension(df_dim_source, table_name, keys, id_col)
                loaded_dims[table_name] = {"df": df_dim_updated}
            else:
                print(f"⚠️ Pulando dimensão {table_name}: Chaves {keys} incompletas.")

        # 2. Fato
        if "fact" in config:
            fact_conf = config["fact"]
            print(f"📊 Fato: {fact_conf['target_table']}")
            df_fact = self._apply_mapping(df_clean, fact_conf["mapping"])
            
            id_col = fact_conf.get("id_col")
            if id_col:
                if id_col in df_fact.columns: df_fact = df_fact.drop(id_col)
                df_fact = df_fact.withColumn(id_col, monotonically_increasing_id())

            for dim_table, info in fact_conf.get("lookups", {}).items():
                if dim_table in loaded_dims:
                    df_dim = loaded_dims[dim_table]["df"]
                    common_cols = [c for c in df_fact.columns if c in df_dim.columns and c != info["target_id_col"]]
                    pk_dim = info.get("pk_dim", "ID")

                    if common_cols:
                        df_dim_sub = df_dim.select(*common_cols, pk_dim)
                        df_fact = df_fact.join(df_dim_sub, on=common_cols, how="left")
                        df_fact = df_fact.withColumnRenamed(pk_dim, info["target_id_col"])
                        df_fact = df_fact.drop(*common_cols)

            mode = fact_conf.get("mode", "append") # Default para append para não apagar histórico
            target_fact_path = f"{self.delta_mgr.base_path}/{fact_conf['target_table']}"
            
            (df_fact.write.format("delta").mode(mode)
             .option("overwriteSchema", "true" if mode=="overwrite" else "false").save(target_fact_path))
            
            self.delta_mgr.log_stats(fact_conf['target_table'])
        
        # Limpar cache do Spark para liberar memória
        self.spark.catalog.clearCache()

class DuckDBLoader:
    def __init__(self, delta_path="data/delta", duckdb_path="data.duckdb", temp_path="data/temp_duck_stage"):
        self.spark = SparkFactory.get_session()
        self.delta_path = delta_path; self.duckdb_path = duckdb_path; self.temp_path = temp_path
        self.EXCLUDE_COLS = ["Nivel_Detalhe"]

    def export_all(self):
        print(f"\n🦆 DuckDB Final Load: {self.duckdb_path}")
        con = duckdb.connect(self.duckdb_path)
        if os.path.exists(self.temp_path): shutil.rmtree(self.temp_path)
        
        if not os.path.exists(self.delta_path):
            print("⚠️ Diretório Delta não encontrado para exportação.")
            return

        tables = [f for f in os.listdir(self.delta_path) if not f.startswith("_") and not f.startswith(".")]
        for table_name in tables:
            print(f"    ⏳ Exportando {table_name}...", end="")
            try:
                df = self.spark.read.format("delta").load(os.path.join(self.delta_path, table_name))
                
                to_drop = [c for c in df.columns if c in self.EXCLUDE_COLS]
                if to_drop: df = df.drop(*to_drop)

                os.makedirs(os.path.join(self.temp_path, table_name), exist_ok=True)
                (df.write.format("parquet").mode("overwrite").save(os.path.join(self.temp_path, table_name)))
                
                con.execute(f'CREATE OR REPLACE TABLE "{table_name}" AS SELECT * FROM read_parquet(\'{self.temp_path}/{table_name}/*.parquet\')')
                count = con.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
                print(f" OK ({count} linhas)")
            except Exception as e: print(f" ERRO: {e}")
        
        con.close()
        if os.path.exists(self.temp_path): shutil.rmtree(self.temp_path)
        print(f"✅ Exportação completa!")

# ==============================================================================
# 2. CONFIGURAÇÕES DE ETL
# ==============================================================================
config_leitos = {
    "name": "ETL_Leitos_Sus",
    "source": { "path": "DYNAMIC", "format": "csv", "options": {}},
    "clean_numbers": ["cnes", "co_cep", "co_ibge"],
    "dimensions": [
        {
            "target_table": "Endereco", "id_col": "Endereco_ID",
            "keys": ["Nivel_Detalhe", "Logradouro", "Numero_do_Logradouro", "CEP"],
            "mapping": {
                "LIT_NIVEL_DETALHE_COMPLETO": "Nivel_Detalhe",
                "regiao": "Regiao_do_Brasil", "uf": "Unidade_Federativa", "co_ibge": "Codigo_do_IBGE",
                "municipio": "Municipio", "no_bairro": "Bairro", "no_logradouro": "Logradouro",
                "nu_endereco": "Numero_do_Logradouro", "no_complemento": "Complemento", "co_cep": "CEP"
            }
        },
        {
            "target_table": "Instituicao", "id_col": "Instituicao_ID", "keys": ["Codigo_CNES"],
            "mapping": {
                "cnes": "Codigo_CNES", "nome_estabelecimento": "Nome_Instituicao", "razao_social": "Razao_Social",
                "tp_gestao": "Tipo_de_Gestao", "co_tipo_unidade": "Codigo_do_Tipo_da_Unidade", "ds_tipo_unidade": "Descricao_do_Tipo_da_Unidade",
                "natureza_juridica": "Codigo_da_Natureza_Juridica", "desc_natureza_juridica": "Descricao_da_Natureza_Juridica",
                "motivo_desabilitacao": "Motivo_da_Desabilitacao", "no_email": "Email", "nu_telefone": "Telefone",
                "LIT_NIVEL_DETALHE_COMPLETO": "Nivel_Detalhe", "no_logradouro": "Logradouro", "nu_endereco": "Numero_do_Logradouro", "co_cep": "CEP"
            },
            "lookups": { "Endereco": {"join_keys": ["Nivel_Detalhe", "Logradouro", "Numero_do_Logradouro", "CEP"], "ref_pk": "Endereco_ID", "target_fk": "Endereco_ID"} }
        }
    ],
    "fact": {
        "target_table": "Leitos", "mode": "append",
        "lookups": { "Instituicao": {"target_id_col": "Instituicao_ID", "pk_dim": "Instituicao_ID"} },
        "mapping": {
            "comp": ("Data_de_Competencia", "date", "yyyyMM"),
            "cnes": "Codigo_CNES",
            "leitos_existentes": ("Quantidade_Leitos_Gerais", "int"),
            "leitos_sus": ("Quantidade_Leitos_SUS", "int"),
            "uti_total_exist": ("Quantidade_Leitos_UTI", "int"),
            "uti_total_sus": ("Quantidade_Leitos_UTI_SUS", "int"),
            "uti_adulto_exist": ("Quantidade_Leitos_UTI_Adulto", "int"), 
            "uti_adulto_sus": ("Quantidade_Leitos_UTI_SUS_Adulto", "int"), 
            "uti_pediatrico_exist": ("Quantidade_Leitos_UTI_Pediatrico", "int"),
            "uti_pediatrico_sus": ("Quantidade_Leitos_UTI_SUS_Pediatrico", "int"),
            "uti_neonatal_exist": ("Quantidade_Leitos_UTI_Neonatal", "int"),
            "uti_neonatal_sus": ("Quantidade_Leitos_UTI_SUS_Neonatal", "int"),
            "uti_queimado_exist": ("Quantidade_Leitos_UTI_Queimado", "int"),
            "uti_queimado_sus": ("Quantidade_Leitos_UTI_SUS_Queimado", "int"),
            "uti_coronariana_exist": ("Quantidade_Leitos_UTI_Coronariana", "int"),
            "uti_coronariana_sus": ("Quantidade_Leitos_UTI_SUS_Coronariana", "int")
        }
    }
}
config_bps = {
    "name": "ETL_BPS_Compras",
    "source": { "path": "DYNAMIC", "format": "csv", "options": {}},
    "clean_numbers": ["cnpj_instituicao", "cnpj_fornecedor", "cnpj_fabricante"],
    "dimensions": [
        {
            "target_table": "Endereco", "id_col": "Endereco_ID",
            "keys": ["Nivel_Detalhe", "Municipio", "Unidade_Federativa"],
            "mapping": {
                "LIT_NIVEL_DETALHE_SIMPLES": "Nivel_Detalhe", "municipio_instituicao": "Municipio", "uf": "Unidade_Federativa",
                "LIT_STRING_VAZIA_LOG": "Logradouro", "LIT_STRING_VAZIA_NUM": "Numero_do_Logradouro", "LIT_STRING_VAZIA_CEP": "CEP",
                "LIT_STRING_VAZIA_COMP": "Complemento", "LIT_STRING_VAZIA_BAIRRO": "Bairro", "LIT_STRING_VAZIA_REGIAO": "Regiao_do_Brasil", "LIT_STRING_VAZIA_IBGE": "Codigo_do_IBGE",
            }
        },
        { "target_table": "Fornecedor", "id_col": "Fornecedor_ID", "keys": ["CNPJ_Fornecedor"], "mapping": {"cnpj_fornecedor": "CNPJ_Fornecedor", "fornecedor": "Nome_Fornecedor"} },
        { "target_table": "Fabricante", "id_col": "Fabricante_ID", "keys": ["CNPJ_Fabricante"], "mapping": {"cnpj_fabricante": "CNPJ_Fabricante", "fabricante": "Nome_Fabricante"} },
        { "target_table": "Produto", "id_col": "Produto_ID", "keys": ["Codigo_CATMAT", "Anvisa"], "mapping": {"codigo_br": "Codigo_CATMAT", "anvisa": "Anvisa", "descricao_catmat": "Descricao_CATMAT", "generico": "Generico"} },
        {
            "target_table": "Instituicao", "id_col": "Instituicao_ID", "keys": ["CNPJ_Instituicao"],
            "mapping": {
                "cnpj_instituicao": "CNPJ_Instituicao", "nome_instituicao": "Nome_Instituicao",
                "LIT_NIVEL_DETALHE_SIMPLES": "Nivel_Detalhe", "municipio_instituicao": "Municipio", "uf": "Unidade_Federativa"
            },
            "lookups": { "Endereco": {"join_keys": ["Nivel_Detalhe", "Municipio", "Unidade_Federativa"], "ref_pk": "Endereco_ID", "target_fk": "Endereco_ID"} }
        }
    ],
    "fact": {
        "target_table": "Instituicao_Compra_Produto", "mode": "append", "id_col": "Instituicao_Compra_Produto_ID",
        "lookups": {
            "Fornecedor": {"target_id_col": "Fornecedor_ID", "pk_dim": "Fornecedor_ID"},
            "Fabricante": {"target_id_col": "Fabricante_ID", "pk_dim": "Fabricante_ID"},
            "Produto": {"target_id_col": "Produto_ID", "pk_dim": "Produto_ID"},
            "Instituicao": {"target_id_col": "Instituicao_ID", "pk_dim": "Instituicao_ID"}
        },
        "mapping": {
            "LIT_STRING_VAZIA_ID": "Instituicao_Compra_Produto_ID", "compra": ("Data_de_Compra", "date", "yyyy/MM/dd HH:mm:ss.SSS"), "insercao": ("Data_de_Insercao", "date", "yyyy/MM/dd HH:mm:ss.SSS"),
            "modalidade_compra": "Modalidade_de_Compra", "capacidade": ("Capacidade", "decimal"), "unidade_medida": "Unidade_de_Medida", "tipo_compra": "Tipo_da_Compra", "qtd_itens_comprados": ("Quantidade_de_Itens", "decimal"), "preco_unitario": ("Preco_Unitario", "decimal"), "preco_total": ("Preco_Total", "decimal"),
            "unidade_fornecimento": "Unidade_de_Fornecimento", "unidade_fornecimento_capacidade": "Capacidade_da_Unidade_de_Fornecimento", "cnpj_fabricante": "CNPJ_Fabricante", "cnpj_fornecedor": "CNPJ_Fornecedor", "codigo_br": "Codigo_CATMAT", "anvisa": "Anvisa", "cnpj_instituicao": "CNPJ_Instituicao"
        }
    }
}

config_bnafar = {
    "name": "ETL_BNAFAR_Estoque",
    "source": { "path": "DYNAMIC", "format": "csv", "options": {}},
    "clean_numbers": ["co_cnes", "co_cep"],
    "dimensions": [
        {
            "target_table": "Endereco", "id_col": "Endereco_ID", "keys": ["Nivel_Detalhe", "Logradouro", "Numero_do_Logradouro", "CEP"],
            "mapping": {
                "LIT_NIVEL_DETALHE_COMPLETO": "Nivel_Detalhe", "no_municipio": "Municipio", "sg_uf": "Unidade_Federativa", "no_logradouro": "Logradouro", "nu_endereco": "Numero_do_Logradouro", "no_bairro": "Bairro", "co_cep": "CEP",
                "nu_latitude": ("Latitude", "latlong"), "nu_longitude": ("Longitude", "latlong")
            }
        },
        {
            "target_table": "Instituicao", "id_col": "Instituicao_ID", "keys": ["Codigo_CNES"],
            "mapping": {
                "co_cnes": "Codigo_CNES", "no_fantasia": "Nome_Instituicao", "no_razao_social": "Razao_Social", "no_email": "Email", "nu_telefone": "Telefone",
                "LIT_NIVEL_DETALHE_COMPLETO": "Nivel_Detalhe", "no_logradouro": "Logradouro", "nu_endereco": "Numero_do_Logradouro", "co_cep": "CEP"
            },
            "lookups": { "Endereco": {"join_keys": ["Nivel_Detalhe", "Logradouro", "Numero_do_Logradouro", "CEP"], "ref_pk": "Endereco_ID", "target_fk": "Endereco_ID"} }
        },
        {
            "target_table": "Produto",
            "id_col": "Produto_ID",
            "keys": ["Codigo_CATMAT", "Anvisa"],
            "mapping": {
                "co_catmat": "Codigo_CATMAT",
                "ds_produto": "Descricao_CATMAT",
                "LIT_NAO_SE_APLICA": "Anvisa"
            }
        }
    ],
    "fact": {
        "target_table": "Instituicao_Estoca_Produto", "mode": "append", "id_col": "Instituicao_Estoca_Produto_ID",
        "lookups": {
            "Instituicao": {"target_id_col": "Instituicao_ID", "pk_dim": "Instituicao_ID"},
            "Produto": {"target_id_col": "Produto_ID", "pk_dim": "Produto_ID"}
        },
        "mapping": {
            "co_cnes": "Codigo_CNES",
            "co_catmat": "Codigo_CATMAT",
            "LIT_NAO_SE_APLICA": "Anvisa",
            "dt_posicao_estoque": ("Data_de_Posicao_no_Estoque", "date", "yyyy/MM/dd"),
            "qt_estoque": ("Quantidade_do_Item_em_Estoque", "decimal"),
            "nu_lote": "Numero_do_Lote",
            "dt_validade": ("Data_de_Validade", "timestamp", "yyyy-MM-dd HH:mm:ssX"),
            "tp_produto": "Tipo_do_Produto",
            "sg_programa_saude": "Sigla_do_Programa_de_Saude",
            "ds_programa_saude": "Descricao_do_Programa_de_Saude",
            "sg_origem": "Sigla_do_Sistema_de_Origem"
        }
    }
}
# ==============================================================================
# 3. MÓDULO WEB SCRAPER & PANDAS CLEANER
# ==============================================================================
def import_unicodedata(s):
    return unicodedata.normalize('NFKD', str(s))
def import_unicodedata_combining(c):
    return unicodedata.combining(c)

class DataFetcher:
    def __init__(self, output_temp_file="temp_stage.csv"):
        self.output_temp_file = output_temp_file

    def fetch_page(self, url):
        try:
            logging.info(f"🔎 Acessando: {url}")
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            return soup(resp.content, "html.parser")
        except Exception as e:
            logging.error(f"❌ Erro ao acessar {url}: {e}")
            return None

    def read_and_clean_csv(self, file_bytes):
        possibilities = [(';', 'utf-8'), (',', 'utf-8'), (';', 'latin-1'), (',', 'latin-1'), (';', 'cp1252')]
        
        for sep, enc in possibilities:
            try:
                file_bytes.seek(0)
                # Testa com poucas linhas
                df = pd.read_csv(file_bytes, sep=sep, encoding=enc, dtype=str, on_bad_lines='skip', nrows=50)
                if len(df.columns) > 1:
                    # Lê completo
                    file_bytes.seek(0)
                    df = pd.read_csv(file_bytes, sep=sep, encoding=enc, dtype=str, on_bad_lines='skip')
                    logging.info(f"✅ CSV detectado: sep='{sep}', enc='{enc}', linhas={len(df)}")
                    return df
            except: continue
        return None

    def download_and_save(self, url, required_columns=None):
        logging.info(f"⬇️ Baixando: {url}")
        try:
            resp = requests.get(url, timeout=180)
            file_obj = io.BytesIO(resp.content)
            
            df = None
            if url.lower().endswith('.zip'):
                with zipfile.ZipFile(file_obj) as zf:
                    csvs = [f for f in zf.namelist() if f.lower().endswith('.csv')]
                    if csvs: df = self.read_and_clean_csv(io.BytesIO(zf.read(csvs[0])))
            else:
                df = self.read_and_clean_csv(file_obj)

            if df is not None and not df.empty:
                # Normaliza colunas
                df.columns = [
                    re.sub(r'[^a-z0-9]+', '_',
                           "".join([c for c in import_unicodedata(col_name) if not import_unicodedata_combining(c)]).lower()
                    ).strip('_')
                    for col_name in df.columns
                ]
                
                # Harmoniza colunas faltantes
                if required_columns:
                    for req in required_columns:
                        if req not in df.columns and not req.startswith("lit_"):
                            df[req] = ""
                            logging.warning(f"⚠️ Coluna esperada '{req}' não encontrada. Criando vazia.")

                df.to_csv(self.output_temp_file, index=False, sep=";", encoding="utf-8")
                return True
            else:
                logging.warning("⚠️ Falha ao ler DataFrame ou arquivo vazio.")
                return False

        except Exception as e:
            logging.error(f"❌ Erro no download/processamento: {e}")
            return False

# ==============================================================================
# 4. ORQUESTRADOR PRINCIPAL
# ==============================================================================
DATASET_URLS = {
    "bnafar": "https://opendatasus.saude.gov.br/dataset/bnafar-posicao-de-estoque",
    "hospitais_leitos": "https://opendatasus.saude.gov.br/dataset/hospitais-e-leitos",
    "bps": "https://opendatasus.saude.gov.br/dataset/bps",
}

CONFIG_MAP = {
    "hospitais_leitos": config_leitos,
    "bps": config_bps,
    "bnafar": config_bnafar
}

def extract_required_columns(config):
    cols = set()
    if "fact" in config:
        for k in config["fact"]["mapping"].keys(): cols.add(k)
    for dim in config.get("dimensions", []):
        for k in dim["mapping"].keys(): cols.add(k)
    return list(cols)

def get_year_from_url(url):
    match = re.search(r'(\d{4})', url)
    if match: return int(match.group(1))
    return 0

def main():
    # NÃO APAGAMOS MAIS A PASTA DELTA
    os.makedirs("data/delta", exist_ok=True)
    
    # Inicializa rastreador de progresso
    tracker = ProcessTracker()
    
    engine = GenericETLEngine(base_output_dir="data/delta")
    fetcher = DataFetcher(output_temp_file="temp_stage.csv")

    for key, url in DATASET_URLS.items():
        logging.info(f"\n{'='*50}\n🔎 Dataset: {key}\n{'='*50}")
        
        config = CONFIG_MAP.get(key)
        if not config: continue

        req_cols = extract_required_columns(config)
        page = fetcher.fetch_page(url)
        if not page: continue

        resources = page.find_all("li", class_="resource-item")
        
        valid_urls = []
        for res in resources:
            link = res.find("a", class_="resource-url-analytics")
            if not link: continue
            file_url = link['href']
            
            if file_url.endswith('.csv') or file_url.endswith('.zip'):
                full_url = urljoin(url, file_url)
                valid_urls.append(full_url)

        valid_urls.sort(key=lambda x: get_year_from_url(x), reverse=False)
        logging.info(f"📅 Ordem de processamento definida: {[get_year_from_url(u) for u in valid_urls]}")

        for file_url in valid_urls:
            # PULA SE JÁ FOI FEITO
            if tracker.is_processed(file_url):
                print(f"⏩ Pulando arquivo já processado: {os.path.basename(file_url)}")
                continue

            # Processa
            success = fetcher.download_and_save(file_url, required_columns=req_cols)
            
            if success:
                try:
                    config['source']['path'] = "temp_stage.csv"
                    engine.run(config)
                    # SUCESSO: Marca no arquivo de texto
                    tracker.mark_processed(file_url)
                except Exception as e:
                    logging.error(f"🔥 Erro no Spark para {file_url}: {e}")
                    # NÃO marca como processado para tentar de novo na próxima
                
                if os.path.exists("temp_stage.csv"): os.remove("temp_stage.csv")
            
            # Limpeza forçada de memória para não estourar o Java Heap
            gc.collect()
            time.sleep(2)

    # 4. Exportação Final
    try:
        loader = DuckDBLoader()
        loader.export_all()
    except Exception as e:
        print(f"Erro na exportação DuckDB: {e}")
    
if __name__ == "__main__":
    main()

🚀 Iniciando nova SparkSession com configurações otimizadas...


25/12/11 19:25:29 WARN Utils: Your hostname, cuba resolves to a loopback address: 127.0.1.1; using 150.164.2.13 instead (on interface enp1s0f0)
25/12/11 19:25:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/datalake_datasus/.cache/pypoetry/virtualenvs/datasus-2Ffj5OkQ-py3.10/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/datalake_datasus/.ivy2/cache
The jars for the packages stored in: /home/datalake_datasus/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5d315455-39d2-4b0a-83e6-8c517ca49cf8;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 256ms :: artifacts dl 11ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default 

⏩ Pulando arquivo já processado: Posicao_Estoque_21-12-2024.zip
⏩ Pulando arquivo já processado: Posicao_Estoque_06-12-2024.zip
⏩ Pulando arquivo já processado: Posicao_Estoque_21-11-2024.zip
⏩ Pulando arquivo já processado: Posicao_Estoque_06-11-2024.zip
⏩ Pulando arquivo já processado: Posicao_Estoque_21-10-2024.zip
⏩ Pulando arquivo já processado: Posicao_Estoque_06-10-2024.zip
⏩ Pulando arquivo já processado: Posicao_Estoque_21-09-2024.zip
⏩ Pulando arquivo já processado: Posicao_Estoque_06-09-2024.zip
⏩ Pulando arquivo já processado: Posicao_Estoque_21-08-2024.zip
⏩ Pulando arquivo já processado: Posicao_Estoque_06-08-2024.zip
⏩ Pulando arquivo já processado: Posicao_Estoque_21-07-2024.zip
⏩ Pulando arquivo já processado: Posicao_Estoque_06-07-2024.zip
⏩ Pulando arquivo já processado: Posicao_Estoque_21-06-2024.zip
⏩ Pulando arquivo já processado: Posicao_Estoque_06-06-2024.zip
⏩ Pulando arquivo já processado: Posicao_Estoque_21-05-2024.zip
⏩ Pulando arquivo já processado: Posicao

...

2025-12-11 20:58:26,108 - INFO - ⬇️ Baixando: https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/BPS/json/2025.json.zip
2025-12-11 20:58:26,365 - WARNING - ⚠️ Falha ao ler DataFrame ou arquivo vazio.
2025-12-11 20:58:28,436 - INFO - ⬇️ Baixando: https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/BPS/xml/2025.xml.zip
2025-12-11 20:58:28,736 - WARNING - ⚠️ Falha ao ler DataFrame ou arquivo vazio.


⚠️ Parando SparkSession existente para aplicar novas configurações...
🚀 Iniciando nova SparkSession com configurações otimizadas...

🦆 DuckDB Final Load: data.duckdb
    ⏳ Exportando Leitos...

 OK (1568612 linhas)
    ⏳ Exportando Fabricante... OK (2792 linhas)
    ⏳ Exportando Instituicao_Compra_Produto...

 OK (263562 linhas)
    ⏳ Exportando Endereco... OK (40716 linhas)
    ⏳ Exportando Instituicao... OK (32460 linhas)
    ⏳ Exportando Fornecedor... OK (2951 linhas)
    ⏳ Exportando Produto... OK (56806 linhas)
    ⏳ Exportando Instituicao_Estoca_Produto...

 OK (219475175 linhas)
✅ Exportação completa!
